In [ ]:
# ── Cell 1: Install ───────────────────────────────────────────────────────────
!pip install -q pymupdf groq python-docx fastapi uvicorn nest_asyncio fitz tools

# ── Cell 2: Imports + Config ──────────────────────────────────────────────────
import io, json, time, asyncio, subprocess, threading, re
import fitz
import uvicorn
import nest_asyncio
from groq import Groq
from docx import Document
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.responses import JSONResponse

GROQ_API_KEY = ""
PORT = 8000

try:
    from kaggle_secrets import UserSecretsClient
    GROQ_API_KEY = UserSecretsClient().get_secret("GROQ_API_KEY")
except Exception:
    pass

client = Groq(api_key=GROQ_API_KEY)

# ── Cell 3: Extractors ────────────────────────────────────────────────────────
def extract_pdf(data: bytes) -> str:
    doc  = fitz.open(stream=data, filetype="pdf")
    text = "".join(page.get_text() for page in doc)
    doc.close()
    return text.strip()

def extract_docx(data: bytes) -> str:
    docx = Document(io.BytesIO(data))
    return "\n".join(p.text for p in docx.paragraphs).strip()

def extract_file(filename: str, data: bytes) -> str:
    if filename.endswith(".pdf"):
        return extract_pdf(data)
    elif filename.endswith(".docx"):
        return extract_docx(data)
    else:
        raise ValueError(f"Unsupported format: {filename}. Use .pdf or .docx")

# ── Cell 4: Chunker ───────────────────────────────────────────────────────────
def recursive_chunk(text, chunk_size=1500, overlap=150, separators=None):
    if separators is None:
        separators = ["\n\n", "\n", ". ", " ", ""]

    if len(text) <= chunk_size:
        return [text.strip()] if text.strip() else []

    for sep in separators:
        if sep == "":
            chunks, start = [], 0
            while start < len(text):
                chunks.append(text[start:start+chunk_size].strip())
                start += chunk_size - overlap
            return [c for c in chunks if c]

        if sep in text:
            parts, chunks, current = text.split(sep), [], ""
            for part in parts:
                candidate = current + sep + part if current else part
                if len(candidate) <= chunk_size:
                    current = candidate
                else:
                    if current:
                        chunks.append(current.strip())
                    if len(part) > chunk_size:
                        sub = recursive_chunk(part, chunk_size, overlap,
                                              separators[separators.index(sep)+1:])
                        if chunks and sub:
                            sub[0] = chunks[-1][-overlap:] + " " + sub[0]
                        chunks.extend(sub)
                        current = ""
                    else:
                        current = part
            if current:
                chunks.append(current.strip())
            return [c for c in chunks if c]

    return [text.strip()]

# ── Cell 5: Groq parser ───────────────────────────────────────────────────────
SYSTEM_PROMPT = """
You are an expert CV/Resume parser.
Extract structured information from CV text and return ONLY valid JSON.
No explanation. No markdown. No backticks. Just raw JSON.

Rules:
- Return null for missing string fields, [] for missing arrays, 0 for missing numbers
- For years_approx: calculate from the duration string (e.g. "Jan 2022 - Dec 2023" = 2).
  If duration says "Present" use 2025 as end year. If completely unclear, return 1.
- Extract ALL skills mentioned anywhere, not just from a skills section
- Keep highlights concise — max 1-2 lines per bullet
- Do NOT hallucinate or infer anything not present in the text
"""

def parse_chunk(chunk: str, retries=2) -> dict:
    prompt = f"""
Extract all available CV information from this text segment.
Return a JSON object with these fields (leave empty if not found):

{{
  "personal": {{
    "name": "", "email": "", "phone": "",
    "location": "", "linkedin": "", "github": "", "portfolio": ""
  }},
  "education": [
    {{"degree": "", "field": "", "institution": "", "year": ""}}
  ],
  "experience": [
    {{
      "role": "", "company": "", "duration": "",
      "years_approx": 0, "highlights": []
    }}
  ],
  "skills": {{
    "technical": [], "tools_and_frameworks": [],
    "soft_skills": [], "languages": []
  }},
  "projects": [
    {{"name": "", "description": "", "tech_stack": [], "impact": ""}}
  ],
  "certifications": []
}}

CV Segment:
---
{chunk}
---

Return only JSON. No markdown. No explanation.
"""
    for attempt in range(retries + 1):
        try:
            resp = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": prompt}
                ],
                temperature=0.1,
                max_tokens=1500,
            )
            raw = resp.choices[0].message.content.strip()
            raw = raw.replace("```json", "").replace("```", "").strip()
            return json.loads(raw)
        except json.JSONDecodeError as e:
            print(f"  JSON error (attempt {attempt+1}): {e}")
            if attempt < retries: time.sleep(1)
        except Exception as e:
            print(f"  Groq error (attempt {attempt+1}): {e}")
            if attempt < retries: time.sleep(2)
    return {}

# ── Cell 6: Merge + Enrich ────────────────────────────────────────────────────
def merge(results: list) -> dict:
    merged = {
        "personal":       {},
        "education":      [],
        "experience":     [],
        "skills":         {"technical": [], "tools_and_frameworks": [], "soft_skills": [], "languages": []},
        "projects":       [],
        "certifications": []
    }
    for r in results:
        if not r: continue
        for k, v in r.get("personal", {}).items():
            if v and not merged["personal"].get(k):
                merged["personal"][k] = v
        for sec in ["education", "experience", "projects", "certifications"]:
            items = r.get(sec, [])
            if not isinstance(items, list): continue
            seen = [json.dumps(e, sort_keys=True) for e in merged[sec]]
            for item in items:
                s = json.dumps(item, sort_keys=True)
                if item and s not in seen:
                    merged[sec].append(item)
                    seen.append(s)
        for cat in ["technical", "tools_and_frameworks", "soft_skills", "languages"]:
            existing = {x.lower() for x in merged["skills"][cat]}
            for skill in r.get("skills", {}).get(cat, []):
                if skill and skill.lower() not in existing:
                    merged["skills"][cat].append(skill)
                    existing.add(skill.lower())
    return merged

def enrich(merged: dict) -> dict:
    total = sum(
        (e.get("years_approx") or (1 if e.get("duration") else 0))
        for e in merged.get("experience", [])
    )
    merged["total_experience_years"] = round(total, 1)
    merged["seniority_level"] = (
        "junior" if total <= 1 else
        "mid"    if total <= 4 else
        "senior" if total <= 8 else "lead"
    )
    name      = merged.get("personal", {}).get("name", "Candidate")
    role      = merged["experience"][0]["role"] if merged["experience"] else "Professional"
    skills    = merged["skills"]["technical"][:3]
    skill_str = ", ".join(skills) if skills else "various technologies"
    merged["summary_one_liner"] = (
        f"{name} is a {merged['seniority_level']}-level {role} "
        f"with {total} year(s) of experience in {skill_str}."
    )
    return merged

# ── Cell 7: Pipeline ──────────────────────────────────────────────────────────
def parse_cv(filename: str, data: bytes) -> dict:
    print(f"→ Extracting: {filename}")
    text = extract_file(filename, data)
    print(f"  {len(text)} chars extracted")
    chunks = recursive_chunk(text)
    print(f"  {len(chunks)} chunk(s)")
    results = []
    for i, chunk in enumerate(chunks):
        print(f"  Chunk {i+1}/{len(chunks)}...")
        results.append(parse_chunk(chunk))
        time.sleep(0.3)
    return enrich(merge(results))

# ── Cell 8: FastAPI app ───────────────────────────────────────────────────────
app = FastAPI(title="CV Parser API")

@app.get("/health")
def health():
    return {"status": "ok", "model": "llama-3.3-70b-versatile"}

@app.post("/parse")
async def parse_endpoint(file: UploadFile = File(..., description="Upload CV (.pdf or .docx)")):
    data = await file.read()
    if not data:
        raise HTTPException(400, "Empty file")
    try:
        profile = parse_cv(file.filename, data)
    except ValueError as e:
        raise HTTPException(400, str(e))
    except Exception as e:
        raise HTTPException(500, f"Parsing failed: {e}")
    return JSONResponse(profile)

# ── Cell 9: Start localhost.run tunnel (no account needed) ───────────────────
def start_tunnel(port: int):
    proc = subprocess.Popen(
        ["ssh", "-o", "StrictHostKeyChecking=no",
         "-R", f"80:localhost:{port}",
         "nokey@localhost.run"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    for line in proc.stdout:
        line = line.decode("utf-8", errors="ignore").strip()
        match = re.search(r"https://[a-zA-Z0-9\-]+\.lhr\.life", line)
        if match:
            url = match.group(0)
            print("\n" + "="*55)
            print(f"  API URL  ->  {url}")
            print(f"  Docs     ->  {url}/docs")
            print(f"  Health   ->  {url}/health")
            print("="*55 + "\n")
            break

threading.Thread(target=start_tunnel, args=(PORT,), daemon=True).start()
time.sleep(5)  # wait for SSH tunnel to establish

# ── Cell 10: Start server (blocks) ───────────────────────────────────────────
nest_asyncio.apply()
config = uvicorn.Config(app, host="0.0.0.0", port=PORT)
server = uvicorn.Server(config)
asyncio.get_event_loop().run_until_complete(server.serve())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 67.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 86.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.7/110.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.9/425.9 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 91.9 MB/s eta 0:00:00:00:01

  API URL  ->  https://395711cdf7f89a.lhr.life
  Docs     ->  https://395711cdf7f89a.lhr.life/docs
  Health   ->  https://395711cdf7f89a.lhr.life/health



INFO:     Started server process [55]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     38.183.101.34:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     38.183.101.34:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     38.183.101.34:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     38.183.101.34:0 - "GET /openapi.json HTTP/1.1" 200 OK
